# Stage 2 — Fine-tuning (Swin UNETR on BraTS-PED)
**Project:** ContextualDenoising-BrainTumorSSL  
**Goal:** Transfer pretrained encoder → 2D Swin UNETR → SOTA Dice on BraTS-PED

### Session checklist
- [ ] GPU accelerator enabled (Settings → Accelerator → GPU T4 x2)
- [ ] Internet ON  ← needed for git clone
- [ ] Datasets attached:
  - `ashfakazadnafi/brats2021-tar`  (contains BraTS-PED subfolder)
  - `pretrained-weights`            (your uploaded pretrain_encoder.pth)

### Before running
Make sure you have uploaded `pretrain_encoder.pth` (output of notebook 01)  
as a Kaggle Dataset named **`pretrained-weights`**.

## 0. Install dependencies

In [ ]:
%%capture
!pip install monai nibabel scikit-image scipy pyyaml tqdm matplotlib

## 1. Clone GitHub repo

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/Nafi14078/ContextualDenoising-BrainTumorSSL.git'
REPO_NAME = 'ContextualDenoising-BrainTumorSSL'
PROJECT   = 'BrainTumor_SSL_Project'

if os.path.exists(f'/kaggle/working/{REPO_NAME}'):
    !rm -rf /kaggle/working/{REPO_NAME}

!git clone {REPO_URL} /kaggle/working/{REPO_NAME}

PROJECT_ROOT = f'/kaggle/working/{REPO_NAME}/{PROJECT}'
sys.path.insert(0, PROJECT_ROOT)

print(f'✓ Repo cloned')
!ls {PROJECT_ROOT}

## 2. Define all paths

In [ ]:
import os

# ── BraTS-PED root (parent of all BraTS-PED-XXXXX folders) ──
BRATSPED_ROOT = (
    '/kaggle/input/datasets/ashfakazadnafi/brats2021-tar/'
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData/'
    'ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData'
)

# ── Pretrained encoder from Stage 1 ──
PRETRAINED_ENCODER = '/kaggle/input/pretrained-weights/pretrain_encoder.pth'

# ── Output paths ──
SLICES_OUT = '/kaggle/working/bratsped_slices'
CKPT_DIR   = '/kaggle/working/checkpoints/finetune'
os.makedirs(SLICES_OUT, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ── Verify paths ──
assert os.path.exists(BRATSPED_ROOT), f'NOT FOUND: {BRATSPED_ROOT}'
subjects = sorted(os.listdir(BRATSPED_ROOT))
print(f'✓ BraTS-PED root found')
print(f'  Total subjects : {len(subjects)}')
print(f'  First 3        : {subjects[:3]}')

if os.path.exists(PRETRAINED_ENCODER):
    size = os.path.getsize(PRETRAINED_ENCODER) / 1e6
    print(f'\n✓ Pretrained encoder found ({size:.1f} MB)')
else:
    print(f'\n⚠️  Pretrained encoder NOT FOUND at {PRETRAINED_ENCODER}')
    print('   Will train from scratch (still valid for ablation baseline)')

# Verify one subject has all 4 modalities + seg
sample_subj = os.path.join(BRATSPED_ROOT, subjects[0])
print(f'\n  Sample subject files:')
for f in sorted(os.listdir(sample_subj)):
    print(f'    {f}')

## 3. Preprocess BraTS-PED → 2D slices + masks
**Run once.** Upload output as Kaggle Dataset `bratsped-preprocessed`.

In [ ]:
import json

meta_path = f'{SLICES_OUT}/metadata.json'

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    train_n = len(meta['train'])
    val_n   = len(meta['val'])
    print(f'✓ Already preprocessed — skipping')
    print(f'  Train slices : {train_n}')
    print(f'  Val slices   : {val_n}')
else:
    print('Running BraTS-PED preprocessing...')
    !python {PROJECT_ROOT}/scripts/preprocess_bratsped.py \
        --root      {BRATSPED_ROOT} \
        --out       {SLICES_OUT} \
        --skip      10 \
        --min_tumor 0.01

## 4. Visualise a sample slice + mask

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

img_dir  = f'{SLICES_OUT}/train/images'
mask_dir = f'{SLICES_OUT}/train/masks'

# Find a slice that actually contains tumor
fnames = sorted(os.listdir(img_dir))
chosen = None
for fname in fnames:
    mask = np.load(f'{mask_dir}/{fname}')
    if mask.max() > 0:
        chosen = fname
        break

img  = np.load(f'{img_dir}/{chosen}')   # (4, H, W)
mask = np.load(f'{mask_dir}/{chosen}')  # (H, W)

COLORS = {0: [0,0,0], 1: [255,0,0], 2: [0,255,0], 3: [0,0,255]}
rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
for l, c in COLORS.items(): rgb[mask == l] = c

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, mod in enumerate(['T1','T1ce','T2','FLAIR']):
    axes[i].imshow(img[i], cmap='gray')
    axes[i].set_title(mod); axes[i].axis('off')
axes[4].imshow(img[3], cmap='gray')
axes[4].imshow(rgb, alpha=0.5)
axes[4].set_title('FLAIR + Mask')
axes[4].axis('off')
plt.suptitle(f'{chosen}  |  Labels: {np.unique(mask).tolist()}', fontsize=11)
plt.tight_layout()
plt.show()
print(f'Label distribution: { {int(l): int((mask==l).sum()) for l in np.unique(mask)} }')

## 5. Update fine-tuning config with Kaggle paths

In [ ]:
import yaml

cfg_path = f'{PROJECT_ROOT}/configs/finetune_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# ── Override all paths ──
cfg['data']['bratsped_root']         = BRATSPED_ROOT
cfg['data']['output_slices']         = SLICES_OUT
cfg['model']['pretrained_encoder']   = PRETRAINED_ENCODER
cfg['training']['checkpoint_dir']    = CKPT_DIR

updated_cfg_path = '/kaggle/working/finetune_config_kaggle.yaml'
with open(updated_cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✓ Config updated')
print(f'  output_slices      : {cfg["data"]["output_slices"]}')
print(f'  pretrained_encoder : {cfg["model"]["pretrained_encoder"]}')
print(f'  checkpoint_dir     : {cfg["training"]["checkpoint_dir"]}')
print(f'  phase1_epochs      : {cfg["training"]["phase1_epochs"]}')
print(f'  phase2_epochs      : {cfg["training"]["phase2_epochs"]}')
print(f'  batch_size         : {cfg["training"]["batch_size"]}')

## 6. Run Fine-tuning
Two phases run automatically:
- **Phase 1** (30 epochs): encoder frozen, decoder warms up
- **Phase 2** (70 epochs): all layers unfrozen, differential LR

⚠️ If session ends mid-training, set `RESUME_PHASE` and `RESUME_FROM` below.

In [ ]:
# ── Resume settings (leave None if starting fresh) ──
RESUME_FROM  = None   # e.g. '/kaggle/input/finetune-ckpt/phase1_epoch_020.pth'
RESUME_PHASE = None   # 1 or 2

import subprocess, sys

cmd = [
    sys.executable,
    f'{PROJECT_ROOT}/finetuning/train_finetune.py',
    '--config', updated_cfg_path
]
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM, '--resume_phase', str(RESUME_PHASE)]

process = subprocess.Popen(
    cmd, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='')
process.wait()
print(f'\nProcess exited with code: {process.returncode}')

## 7. Evaluate best model — Dice WT / TC / ET + HD95

In [ ]:
import torch, numpy as np, yaml, sys
sys.path.insert(0, PROJECT_ROOT)

from finetuning.swin_unetr_2d import build_model
from finetuning.dataset       import get_finetune_loaders
from evaluation.metrics       import (compute_dice_wt, compute_dice_tc,
                                       compute_dice_et, compute_hd95,
                                       print_metrics)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(updated_cfg_path) as f:
    cfg = yaml.safe_load(f)

# Load best model
best_path = f'{CKPT_DIR}/best_model.pth'
model     = build_model(cfg, device)
ckpt      = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'✓ Loaded best model from epoch {ckpt["epoch"]+1} (Phase {ckpt["phase"]})')

# Val loader
_, val_loader = get_finetune_loaders(
    slices_dir  = cfg['data']['output_slices'],
    patch_size  = cfg['data']['patch_size'],
    batch_size  = cfg['training']['batch_size'],
    num_workers = 2
)

dice_wt, dice_tc, dice_et = [], [], []
hd95_wt, hd95_tc, hd95_et = [], [], []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        masks  = batch['mask']
        preds  = model(images).argmax(dim=1).cpu().numpy()
        gt     = masks.numpy()

        for b in range(preds.shape[0]):
            dice_wt.append(compute_dice_wt(preds[b], gt[b]))
            dice_tc.append(compute_dice_tc(preds[b], gt[b]))
            dice_et.append(compute_dice_et(preds[b], gt[b]))
            hd95_wt.append(compute_hd95(preds[b] > 0,             gt[b] > 0))
            hd95_tc.append(compute_hd95(np.isin(preds[b],[1,3]),  np.isin(gt[b],[1,3])))
            hd95_et.append(compute_hd95(preds[b] == 3,            gt[b] == 3))

results = {
    'Dice WT'   : float(np.mean(dice_wt)),
    'Dice TC'   : float(np.mean(dice_tc)),
    'Dice ET'   : float(np.mean(dice_et)),
    'Mean Dice' : float(np.mean(dice_wt + dice_tc + dice_et)) / 3,
    'HD95 WT'   : float(np.mean(hd95_wt)),
    'HD95 TC'   : float(np.mean(hd95_tc)),
    'HD95 ET'   : float(np.mean(hd95_et)),
}
print_metrics(results, stage='Final Evaluation')

## 8. Visual — segmentation overlay on val slices

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)
from evaluation.visualize import plot_segmentation_overlay

# Show 3 val samples
model.eval()
shown = 0
with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        masks  = batch['mask']
        preds  = model(images).argmax(dim=1).cpu()

        for b in range(images.shape[0]):
            # Only show slices with actual tumor
            if masks[b].max() == 0:
                continue
            plot_segmentation_overlay(
                image     = images[b].cpu(),
                gt_mask   = masks[b],
                pred_mask = preds[b],
                title     = f'Val sample {shown+1}'
            )
            shown += 1
            if shown >= 3:
                break
        if shown >= 3:
            break

## 9. Plot training curves

In [ ]:
# If you logged metrics to a JSON file during training, load and plot here
# Otherwise this cell shows how to manually build the plot
import json, os
from evaluation.visualize import plot_training_curves

log_path = f'{CKPT_DIR}/training_log.json'
if os.path.exists(log_path):
    with open(log_path) as f:
        history = json.load(f)
    plot_training_curves(history, title='Fine-tuning Training Curves')
else:
    print('No training_log.json found.')
    print('Training curves are printed to stdout during training.')